# FABExp Package Downloader

In this notebook, users can select and download the FABExp package they want to use from Github. Run the cell below, select the package name in the dropdown menu and click the **Download Package** button. After the package is downloaded, users can click the **Untar Package** button. Then they can go to the folder and run *welcome.ipynb* to start their experiments.

In [ ]:
import requests
import subprocess
import ipywidgets as widgets

repo_owner = 'fabric-testbed'
repo_name = 'FABExp'
folders = ['demo', 'FABExp_Tested_Packages/fabric_examples', 'FABExp_Tested_Packages/teaching_materials']


# Function to get the list of FABExp package names from github
def get_FABExp_package_names_from_github(repo_owner, repo_name, folders):
    base_url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/contents/"
    package_names = []

    for folder in folders:
        response = requests.get(base_url + folder)
        if response.status_code == 200:
            files = response.json()
            for file in files:
                if file['name'].endswith('.tar.gz'):
                    name = f"{folder}/{file['name']}"
                    package_names.append(name)
        else:
            print(f"Error accessing {folder}: {response.status_code}")

    return package_names

packages = get_FABExp_package_names_from_github(repo_owner, repo_name, folders)

def download_file(b):
    with download_output:
        download_output.clear_output()
        selected_index = dropdown.index
        selected_file = packages[selected_index]
        url = f"https://raw.githubusercontent.com/{repo_owner}/{repo_name}/main/"
        command ="{} {}{}".format('wget', url, selected_file)
        print(f"Downloading the package using: {command}")
        result = subprocess.run(command, shell=True, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        if result.returncode == 0:
            print("Download successful.")
        else:
            print("Download failed.")
            
def untar_file(b):
    with untar_output:
        untar_output.clear_output()
        selected_index = dropdown.index
        selected_file = packages[selected_index]
        f_name = selected_file.split("/")[-1]
        command = f"tar -xvf {f_name}"
        print (f"Running: {command} to untar the package...")
        try:
            result = subprocess.run(command, shell=True, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        except Exception as e:
            print (f"Error: {e}")
        
        
dropdown = widgets.Dropdown(
    options=[p.replace('.tar.gz', '').replace('FABExp_Tested_Packages/', '') for p in packages],
    description='Select the FABExp Package to Download:',
    style={'description_width': 'initial'},
    layout={'width': '75%'} 
)
title = widgets.HTML("<h2 style='color: darkblue;'>FABExp Package Downloader</h2>")
download_button = widgets.Button(description="Download Package")
download_output = widgets.Output()
download_button.on_click(download_file)
untar_button = widgets.Button(description="Untar Package")
untar_output = widgets.Output()
untar_button.on_click(untar_file)


display(title, dropdown, download_button, download_output, untar_button, untar_output)